# Lista 5 - Klasyfikacja wydzwieku (PolEmo2.0-IN): encoder vs decoder

**Przedmiot:** Sztuczna inteligencja i inzynieria wiedzy
**Autor:** Dawid Pilarski

Porownanie dwoch podejsc do analizy wydzwieku na zbiorze `allegro/klej-polemo2-in` (split `test`):

- **encoder-only** (BERT-like, np. HerBERT) - dedykowana glowa klasyfikujaca,
- **decoder-only LLM** (np. Qwen2.5) - klasyfikacja zero-shot przez prompt.

Klasyfikujemy do **3 klas** (`negative`, `neutral`, `positive`) - zgodnie z trescia listy
**wyrzucamy klase `ambiguous`**. Szczegolny nacisk kladziemy na **mapowanie etykiet tekstowych
modelu na numeryczne id zgodne ze zbiorem** (punktowany element listy).

> Cala logika siedzi w pakiecie `src/` (jedno zrodlo prawdy). Notebook tylko ja wola
> i pokazuje wyniki. Parametry zmieniasz w `src/config.py`.

## 0. Setup srodowiska

Wykrywamy Colab, instalujemy zaleznosci i upewniamy sie, ze `src/` jest na sciezce.
W Colab pamietaj o GPU: **Runtime -> Change runtime type -> T4 GPU**.

In [ ]:
# Wykrycie Colab
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Colab:", IN_COLAB)

In [ ]:
# Instalacja zaleznosci (glownie w Colab; lokalnie uzyj requirements.txt)
if IN_COLAB:
    !pip install -q "transformers>=4.44" datasets accelerate \
        langchain-huggingface langchain-core "pydantic>=2" \
        bitsandbytes sentencepiece protobuf scikit-learn matplotlib seaborn tqdm
    # Jezeli load_dataset rzuca blad o skryptach, odkomentuj ponizsze:
    # !pip install -q "datasets<3.0"

In [ ]:
# Zapewniamy dostep do pakietu src/.
# Scenariusze: (a) src/ lezy obok notebooka, (b) wgrywamy zip i rozpakowujemy,
# (c) klonujemy repo. Wybierz to, co pasuje.
import os, sys

def ensure_src():
    # (a) src obok notebooka?
    for base in [".", "lista5_polemo"]:
        if os.path.isdir(os.path.join(base, "src")):
            sys.path.insert(0, os.path.abspath(base))
            return os.path.abspath(base)
    # (b) jest zip w katalogu? rozpakuj
    import glob, zipfile
    zips = glob.glob("*.zip")
    if zips:
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(".")
        for base in [".", "lista5_polemo"]:
            if os.path.isdir(os.path.join(base, "src")):
                sys.path.insert(0, os.path.abspath(base))
                return os.path.abspath(base)
    # (c) w Colab pozwol wgrac zip recznie
    if IN_COLAB:
        from google.colab import files
        print("Wgraj lista5_polemo.zip:")
        up = files.upload()
        import zipfile
        for fn in up:
            if fn.endswith(".zip"):
                with zipfile.ZipFile(fn) as z:
                    z.extractall(".")
        for base in [".", "lista5_polemo"]:
            if os.path.isdir(os.path.join(base, "src")):
                sys.path.insert(0, os.path.abspath(base))
                return os.path.abspath(base)
    raise RuntimeError("Nie znalazlem katalogu src/. Wgraj projekt obok notebooka.")

PROJECT_ROOT = ensure_src()
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
# Importy z naszego pakietu + biblioteki pomocnicze
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from src.utils import set_seed, free_memory, device_info
from src import data, encoder, decoder, metrics
from src.decoder import PROMPTS
from src.config import (
    SEED, MAX_SAMPLES, DEFAULT_ENCODER, ENCODER_MODELS,
    DEFAULT_LLM, LLM_MODELS, USE_4BIT, CLASS_NAMES,
    LLM_MAX_NEW_TOKENS, LLM_JSON_MAX_NEW_TOKENS,
)

set_seed(SEED)
print(device_info())
print("MAX_SAMPLES =", MAX_SAMPLES, "(None = caly zbior; na iteracje ustaw 150-200 w config.py)")

## 1. Zbior danych PolEmo2.0-IN (10 pkt)

Ladujemy split `test`, ogladamy surowe statystyki, wyrzucamy `ambiguous`, mapujemy etykiety
na kanoniczne id i liczymy statystyki dlugosci tekstow.

In [ ]:
ds = data.load_polemo_test()
raw = data.describe_raw(ds)
print("Liczba probek (surowo):", raw["n_total"])
print("Kolumny:", raw["columns"], "| tekst:", raw["text_col"], "| etykieta:", raw["label_col"])
print("Rozklad klas (surowy):", raw["class_counts_raw"])

In [ ]:
# Wyrzucamy ambiguous + mapujemy na id (negative=0, neutral=1, positive=2)
texts, labels, meta = data.prepare_dataset(ds, max_samples=MAX_SAMPLES)
print("Uzyte probki:", meta["n_used"], "| wyrzucono ambiguous:", meta["n_ambiguous_dropped"])

counts = Counter(labels)
print("Rozklad klas (uzyty):")
for cid in sorted(counts):
    print(f"  {CLASS_NAMES[cid]:>8} (id={cid}): {counts[cid]}")

In [ ]:
# Statystyki dlugosci tekstow
tl = data.text_length_stats(texts)
print("Dlugosc [znaki]:", {k: round(v, 1) for k, v in tl["chars"].items()})
print("Dlugosc [slowa]:", {k: round(v, 1) for k, v in tl["words"].items()})

In [ ]:
# Wykresy: rozklad klas + histogram dlugosci (w slowach)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

names = [CLASS_NAMES[c] for c in sorted(counts)]
vals = [counts[c] for c in sorted(counts)]
ax[0].bar(names, vals, color=["#d9534f", "#999999", "#5cb85c"])
ax[0].set_title("Zrownowazenie klas (test, bez ambiguous)")
ax[0].set_ylabel("liczba probek")

ax[1].hist(tl["word_lens"], bins=40, color="#4a90d9", edgecolor="white")
ax[1].set_title("Rozklad dlugosci tekstu (slowa)")
ax[1].set_xlabel("liczba slow")
ax[1].set_ylabel("liczba probek")

plt.tight_layout()
plt.show()

## 2. Encoder baseline - HerBERT (20 pkt)

Model `Voicelab/herbert-base-cased-sentiment` przez `transformers.pipeline`.
Etykiety tekstowe modelu mapujemy na nasze kanoniczne id funkcja `build_model_label_map`
(czyta `model.config.id2label` - **nie hardkodujemy** mapowania).

In [ ]:
pipe, label_map = encoder.load_encoder(DEFAULT_ENCODER)
print("Mapa etykiet modelu -> kanoniczne id:", label_map)

enc_pred = encoder.classify_encoder(pipe, label_map, texts)
m_enc = metrics.compute_metrics(labels, enc_pred)
metrics.print_metrics(DEFAULT_ENCODER, m_enc)

In [ ]:
metrics.plot_confusion(labels, enc_pred, DEFAULT_ENCODER, save=True, show=True)
free_memory(pipe)
del pipe

## 3. Encoder - eksploracja parametrow (20 pkt)

**Uwaga o temperaturze:** temperatura dotyczy *generacji* (samplingu tokenow), a encoder
robi deterministyczny forward pass z softmaxem na glowie klasyfikujacej - tu **nie ma czego
samplowac**. Dlatego eksploracje encodera robimy przez **porownanie roznych modeli**, nie
temperature. (To swiadoma uwaga do raportu, a nie pominiecie.)

In [ ]:
enc_rows = [{
    "model": DEFAULT_ENCODER, "typ": "encoder",
    "accuracy": m_enc["accuracy"], "f1_macro": m_enc["f1_macro"],
    "f1_weighted": m_enc["f1_weighted"],
}]

for name in ENCODER_MODELS:
    if name == DEFAULT_ENCODER:
        continue
    print(f"\n--- {name} ---")
    try:
        pipe, label_map = encoder.load_encoder(name)
        pred = encoder.classify_encoder(pipe, label_map, texts)
        m = metrics.compute_metrics(labels, pred)
        metrics.print_metrics(name, m)
        enc_rows.append({"model": name, "typ": "encoder",
                         "accuracy": m["accuracy"], "f1_macro": m["f1_macro"],
                         "f1_weighted": m["f1_weighted"]})
        del pipe
        free_memory()
    except Exception as e:
        print("BLAD:", e)

metrics.results_table(enc_rows)

## 4. Decoder baseline - Qwen LLM (20 pkt)

Model `Qwen/Qwen2.5-1.5B-Instruct` jako klasyfikator zero-shot. Prosty prompt po angielsku,
**temperatura 0.0** (deterministycznie). Wyjscie modelu parsujemy do kanonicznego id;
gdy sie nie uda - fallback na `neutral` i zliczamy `n_unparsed`.

In [ ]:
model, tok = decoder.load_llm(DEFAULT_LLM, use_4bit=USE_4BIT)

llm_pred, n_unp = decoder.classify_llm(model, tok, texts, PROMPTS["en_basic"], temperature=0.0)
m_llm = metrics.compute_metrics(labels, llm_pred, n_unparsed=n_unp)
metrics.print_metrics(f"{DEFAULT_LLM} (en_basic, t=0.0)", m_llm)
print("Nieparsowalne odpowiedzi:", n_unp, "/", len(texts))

In [ ]:
metrics.plot_confusion(labels, llm_pred, f"{DEFAULT_LLM}_en_basic", save=True, show=True)

## 5. Decoder - eksploracja parametrow (30 pkt)

Sprawdzamy **co najmniej dwa aspekty**:
- **5a** wplyw promptu (en / pl / few-shot / wymuszony JSON),
- **5b** wplyw temperatury,
- **5c** (opcjonalnie) inny model / kwantyzacja 4-bit,
- **5d** parsowanie przez `langchain_core.output_parsers.JsonOutputParser`.

### 5a. Wplyw promptu (temp = 0.0)

In [ ]:
prompt_rows = []
for prompt_name, fn in PROMPTS.items():
    json_mode = prompt_name == "json"
    mnt = LLM_JSON_MAX_NEW_TOKENS if json_mode else LLM_MAX_NEW_TOKENS
    print(f"\n--- prompt: {prompt_name} (json_mode={json_mode}) ---")
    pred, n_unp = decoder.classify_llm(model, tok, texts, fn,
                                       temperature=0.0, max_new_tokens=mnt,
                                       json_mode=json_mode)
    m = metrics.compute_metrics(labels, pred, n_unparsed=n_unp)
    metrics.print_metrics(f"{prompt_name} (t=0.0)", m)
    prompt_rows.append({"prompt": prompt_name, "accuracy": m["accuracy"],
                        "f1_macro": m["f1_macro"], "f1_weighted": m["f1_weighted"],
                        "n_unparsed": n_unp})

pd.DataFrame(prompt_rows).sort_values("f1_macro", ascending=False)

### 5b. Wplyw temperatury (prompt: pl_fewshot)

In [ ]:
temp_rows = []
for temp in [0.0, 0.3, 0.7, 1.0]:
    print(f"\n--- temperatura: {temp} ---")
    pred, n_unp = decoder.classify_llm(model, tok, texts, PROMPTS["pl_fewshot"],
                                       temperature=temp)
    m = metrics.compute_metrics(labels, pred, n_unparsed=n_unp)
    metrics.print_metrics(f"pl_fewshot (t={temp})", m)
    temp_rows.append({"temperatura": temp, "accuracy": m["accuracy"],
                      "f1_macro": m["f1_macro"], "f1_weighted": m["f1_weighted"],
                      "n_unparsed": n_unp})

pd.DataFrame(temp_rows)

### 5c. (Opcjonalnie) Inny model / kwantyzacja 4-bit

Zwalniamy biezacy model i ladujemy wiekszy (`Qwen2.5-3B-Instruct`) w 4-bit.
Wymaga `bitsandbytes`. Jezeli brakuje VRAM - pomin te komorke.

In [ ]:
RUN_5C = False  # ustaw True, zeby porownac wiekszy model w 4-bit

if RUN_5C:
    del model, tok
    free_memory()

    model, tok = decoder.load_llm("Qwen/Qwen2.5-3B-Instruct", use_4bit=True)
    pred, n_unp = decoder.classify_llm(model, tok, texts, PROMPTS["pl_fewshot"], temperature=0.0)
    m = metrics.compute_metrics(labels, pred, n_unparsed=n_unp)
    metrics.print_metrics("Qwen2.5-3B-Instruct 4bit (pl_fewshot, t=0.0)", m)

### 5d. Parsowanie przez LangChain JsonOutputParser

Stos zgodny z lista: `HuggingFacePipeline` + `PromptTemplate` + `JsonOutputParser` (+ `pydantic`).
Dziala one-by-one (wolniej), wiec puszczamy na podzbiorze probek - chodzi o pokazanie mechanizmu.

In [ ]:
N_DEMO = min(50, len(texts))
lc_pred, lc_unp = decoder.langchain_json_demo(DEFAULT_LLM, texts[:N_DEMO], temperature=0.0)
m_lc = metrics.compute_metrics(labels[:N_DEMO], lc_pred, n_unparsed=lc_unp)
metrics.print_metrics(f"LangChain JSON demo (n={N_DEMO})", m_lc)
print("Nieparsowalne:", lc_unp, "/", N_DEMO)

In [ ]:
# Sprzatamy LLM po zadaniu 5
try:
    del model, tok
except NameError:
    pass
free_memory()

## 6. Podsumowanie

Zbieramy wszystko do jednej tabeli i zapisujemy do `outputs/wyniki.csv`.
Najlepszy wynik kazdej rodziny modeli przepisz potem do `RAPORT.md`.

In [ ]:
summary_rows = list(enc_rows)
summary_rows.append({
    "model": f"{DEFAULT_LLM} (en_basic, t=0.0)", "typ": "decoder",
    "accuracy": m_llm["accuracy"], "f1_macro": m_llm["f1_macro"],
    "f1_weighted": m_llm["f1_weighted"],
})

df = metrics.results_table(summary_rows)
os.makedirs("outputs", exist_ok=True)
df.to_csv("outputs/wyniki.csv", index=False)
print("Zapisano outputs/wyniki.csv")
df

---
### Co dalej / na co zwrocic uwage w raporcie

- **Mapowanie etykiet** to punktowany element listy - opisz, jak `build_model_label_map`
  czyta `id2label` modelu i jak `parse_generated_label` radzi sobie z wolnym tekstem LLM.
- **Niezbalansowanie klas** (zwykle przewaga negatywnych) -> dlatego patrzymy na **F1 macro**,
  nie tylko accuracy.
- **Temperatura** poprawia/pogarsza tylko LLM; dla encodera jest bez znaczenia (uzasadnij).
- Jezeli rozklad klas w `test` wyglada zdegenerowanie, sprawdz uwagi w `README.md`
  (sekcja troubleshooting) - byc moze etykiety sa w innym splicie.